In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# Baselines (IDEAL) - baseline02 mean-by-hour, baseline03 mean-by-hour-weekend
# Uses: consumption_Wh (DO NOT rename)
# Saves: overall, per-home, predictions CSVs
# ============================================================

BASE_DIR = Path("C:/IDEAL_Programming")
DATA_PATH = BASE_DIR / "processed" / "final" / "IDEAL_final_hourly_dataset.csv"
OUT_DIR = BASE_DIR / "processed" / "baselines" / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TIME_COL = "timestamp"
HOME_COL = "home_id"
TARGET_COL = "consumption_Wh"

def add_time_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    df["hour"] = df[TIME_COL].dt.hour
    df["dow"] = df[TIME_COL].dt.dayofweek  # Mon=0..Sun=6
    df["is_weekend"] = df["dow"].isin([5, 6]).astype(int)
    return df

def safe_mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # MAPE is unstable when y_true ~ 0, so we compute it only where y_true > 0
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100.0)

def compute_metrics(df_eval: pd.DataFrame) -> dict:
    y = df_eval[TARGET_COL].to_numpy(dtype=float)
    yhat = df_eval["yhat"].to_numpy(dtype=float)

    mae = float(np.mean(np.abs(y - yhat)))
    rmse = float(np.sqrt(np.mean((y - yhat) ** 2)))
    mape = safe_mape(y, yhat)

    return {"MAE": mae, "RMSE": rmse, "MAPE_%": mape, "rows_evaluable": int(len(df_eval))}

def per_home_metrics(df_eval: pd.DataFrame) -> pd.DataFrame:
    g = df_eval.groupby(HOME_COL, dropna=False)
    rows = []
    for home_id, grp in g:
        y = grp[TARGET_COL].to_numpy(dtype=float)
        yhat = grp["yhat"].to_numpy(dtype=float)
        rows.append({
            HOME_COL: home_id,
            "rows_evaluable": int(len(grp)),
            "MAE": float(np.mean(np.abs(y - yhat))),
            "RMSE": float(np.sqrt(np.mean((y - yhat) ** 2))),
            "MAPE_%": safe_mape(y, yhat),
        })
    return pd.DataFrame(rows)

# -----------------------------
# Load data
# -----------------------------
df = pd.read_csv(DATA_PATH, low_memory=False)
df = add_time_cols(df)

# Keep only rows with target
df = df[df[TARGET_COL].notna()].copy()

print(f"Rows total: {len(df)}")
print(f"Unique homes: {df[HOME_COL].nunique()}")

# ============================================================
# Baseline 02: mean-by-hour (global)
# yhat = mean(consumption_Wh | hour)
# ============================================================
hour_means = df.groupby("hour")[TARGET_COL].mean()

df_b02 = df[[HOME_COL, TIME_COL, TARGET_COL, "hour", "is_weekend"]].copy()
df_b02["yhat"] = df_b02["hour"].map(hour_means)

df_b02_eval = df_b02.dropna(subset=["yhat"]).copy()

m02 = compute_metrics(df_b02_eval)
overall02 = pd.DataFrame([{
    "baseline": "mean_by_hour_global",
    **m02
}])
perhome02 = per_home_metrics(df_b02_eval)

pred02 = df_b02_eval[[HOME_COL, TIME_COL, TARGET_COL, "yhat", "hour", "is_weekend"]].copy()
pred02["baseline"] = "baseline02_mean_by_hour_global"

# Save
overall02_path = OUT_DIR / "baseline02_mean_by_hour_overall.csv"
perhome02_path = OUT_DIR / "baseline02_mean_by_hour_per_home.csv"
pred02_path    = OUT_DIR / "baseline02_mean_by_hour_predictions.csv"

overall02.to_csv(overall02_path, index=False)
perhome02.to_csv(perhome02_path, index=False)
pred02.to_csv(pred02_path, index=False)

print("\n=== Baseline02 (mean-by-hour global) ===")
print(overall02)

# ============================================================
# Baseline 03: mean-by-hour × weekend (global)
# yhat = mean(consumption_Wh | hour, is_weekend)
# ============================================================
hw_means = df.groupby(["hour", "is_weekend"])[TARGET_COL].mean()

df_b03 = df[[HOME_COL, TIME_COL, TARGET_COL, "hour", "is_weekend"]].copy()
df_b03["yhat"] = df_b03.apply(lambda r: hw_means.get((r["hour"], r["is_weekend"]), np.nan), axis=1)

df_b03_eval = df_b03.dropna(subset=["yhat"]).copy()

m03 = compute_metrics(df_b03_eval)
overall03 = pd.DataFrame([{
    "baseline": "mean_by_hour_weekend_global",
    **m03
}])
perhome03 = per_home_metrics(df_b03_eval)

pred03 = df_b03_eval[[HOME_COL, TIME_COL, TARGET_COL, "yhat", "hour", "is_weekend"]].copy()
pred03["baseline"] = "baseline03_mean_by_hour_weekend_global"

# Save
overall03_path = OUT_DIR / "baseline03_mean_by_hour_weekend_overall.csv"
perhome03_path = OUT_DIR / "baseline03_mean_by_hour_weekend_per_home.csv"
pred03_path    = OUT_DIR / "baseline03_mean_by_hour_weekend_predictions.csv"

overall03.to_csv(overall03_path, index=False)
perhome03.to_csv(perhome03_path, index=False)
pred03.to_csv(pred03_path, index=False)

print("\n=== Baseline03 (mean-by-hour × weekend global) ===")
print(overall03)

# ============================================================
# Optional: quick combined summary table saved to CSV
# ============================================================
summary = pd.concat([overall02, overall03], ignore_index=True)
summary_path = OUT_DIR / "baseline02_03_summary.csv"
summary.to_csv(summary_path, index=False)

print("\nSaved:")
print(overall02_path)
print(perhome02_path)
print(pred02_path)
print(overall03_path)
print(perhome03_path)
print(pred03_path)
print(summary_path)

print("\nDone.")


Rows total: 1529350
Unique homes: 254

=== Baseline02 (mean-by-hour global) ===
              baseline         MAE        RMSE      MAPE_%  rows_evaluable
0  mean_by_hour_global  254.465677  421.044779  122.856135         1529350

=== Baseline03 (mean-by-hour × weekend global) ===
                      baseline         MAE        RMSE      MAPE_%  \
0  mean_by_hour_weekend_global  253.198212  419.485057  121.783585   

   rows_evaluable  
0         1529350  

Saved:
C:\IDEAL_Programming\processed\baselines\results\baseline02_mean_by_hour_overall.csv
C:\IDEAL_Programming\processed\baselines\results\baseline02_mean_by_hour_per_home.csv
C:\IDEAL_Programming\processed\baselines\results\baseline02_mean_by_hour_predictions.csv
C:\IDEAL_Programming\processed\baselines\results\baseline03_mean_by_hour_weekend_overall.csv
C:\IDEAL_Programming\processed\baselines\results\baseline03_mean_by_hour_weekend_per_home.csv
C:\IDEAL_Programming\processed\baselines\results\baseline03_mean_by_hour_weekend_p